# HTR CREMMA Medieval — Évaluation sur le TEST SCELLÉ

But : mesurer le **CER officiel** d'un modèle sur `test_clean.arrow` (3 documents scellés, mode L).

- À lancer **une seule fois** par modèle final : le test ne doit jamais servir à régler les hyperparamètres.
- Attendre un CER **plus élevé que la validation** : le test contient des manuscrits/langues non vus → il mesure la généralisation réelle.
- Aucun entraînement ici. Juste `ketos test`.

**Prérequis :** Kaggle Secrets `AWS_ACCESS_KEY_ID` + `AWS_SECRET_ACCESS_KEY`.

**Ordre :** 1 → 2 → 3

In [ ]:
# Cellule 1 — Connexion S3
from kaggle_secrets import UserSecretsClient
import boto3

secrets = UserSecretsClient()
s3 = boto3.client(
    's3',
    aws_access_key_id=secrets.get_secret('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=secrets.get_secret('AWS_SECRET_ACCESS_KEY'),
    region_name='eu-west-3'
)
print('Connexion S3 OK')

In [ ]:
# Cellule 2 — Installer Kraken
import subprocess, sys, torch
TORCH_VER = torch.__version__
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'kraken', 'iso639-lang'], check=True)
# editdistance pour le calcul CER manuel si besoin
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'editdistance'], check=True)
print('OK')


In [ ]:
# Cellule 3 — Evaluation CER sur le test scelle
# Contournement du bug safetensors 0.5.x (tenseurs LSTM partages) :
# on reecrit le .safetensors avec les tenseurs departhages avant de le passer a ketos.
import os, io, shutil, struct, json
import pyarrow as pa
from PIL import Image
import torch
from safetensors.torch import load_file, save_file

os.makedirs('/kaggle/working/splits', exist_ok=True)

# --- Modele a evaluer ---
#   best_0.7367_20260613_1738.safetensors        (Run 4 — meilleur en validation)
#   exp3_clean_arrow_20260613_2252.safetensors    (Exp 3 grayscale)
#   exp3opt_finetune_20260615_1508.safetensors
#   exp3opt_finetune_20260615_1849.safetensors    (dernier)
MODELE_S3    = 'output/exp3_clean_arrow_20260613_2252.safetensors'
MODELE_LOCAL = '/kaggle/working/modele_orig.safetensors'
MODELE_FIXED = '/kaggle/working/modele_fixed.safetensors'
TEST_LOCAL   = '/kaggle/working/splits/test_clean.arrow'

# Telecharger
s3.download_file('htr-cremma-medieval', 'splits/test_clean.arrow', TEST_LOCAL)
s3.download_file('htr-cremma-medieval', MODELE_S3, MODELE_LOCAL)
print(f'Modele telecharge : {os.path.getsize(MODELE_LOCAL)/1024/1024:.1f} MB')

# Verifier mode L du test
with pa.memory_map(TEST_LOCAL, 'rb') as src:
    t = pa.ipc.open_file(src).read_all()
col = 'lines' if 'lines' in t.column_names else t.column_names[0]
modes = {}
for i in range(t.num_rows):
    rec = t.column(col)[i].as_py()
    raw = rec['im'] if isinstance(rec, dict) else rec
    m = Image.open(io.BytesIO(raw)).mode
    modes[m] = modes.get(m, 0) + 1
print(f'test_clean.arrow : {t.num_rows} lignes, modes {modes}')

# --- Patch safetensors : cloner chaque tenseur pour casser le partage memoire ---
# safetensors 0.5+ refuse de charger des poids LSTM qui partagent le meme storage.
# .clone() cree une copie independante -> plus de tenseurs partages -> save_file accepte.
def lire_metadata_safetensors(path):
    with open(path, 'rb') as f:
        n = struct.unpack('<Q', f.read(8))[0]
        header = json.loads(f.read(n).decode('utf-8'))
    return {k: v for k, v in header.get('__metadata__', {}).items()}

print('Patch safetensors (clone tenseurs)...')
tensors_orig = load_file(MODELE_LOCAL, device='cpu')
tensors_fixed = {k: v.clone().contiguous() for k, v in tensors_orig.items()}
metadata = lire_metadata_safetensors(MODELE_LOCAL)
save_file(tensors_fixed, MODELE_FIXED, metadata=metadata)
print(f'Modele patche : {os.path.getsize(MODELE_FIXED)/1024/1024:.1f} MB')

# --- ketos test sur le modele patche ---
import subprocess, shutil
KETOS_BIN = shutil.which('ketos') or '/usr/local/bin/ketos'
env = os.environ.copy()
env['PYTHONIOENCODING'] = 'utf-8'
env['PYTHONUTF8'] = '1'
env['LC_ALL'] = 'C.UTF-8'

LOG = '/kaggle/working/ketos_test.log'
cmd = [KETOS_BIN, 'test', '-m', MODELE_FIXED, '-f', 'binary', '-u', 'NFD', TEST_LOCAL]
print('Commande :', ' '.join(cmd))
print('-' * 60)

with open(LOG, 'wb') as logf:
    rc = subprocess.call(cmd, env=env, stdout=logf, stderr=subprocess.STDOUT)

raw = open(LOG, 'rb').read().decode('utf-8', errors='replace')
print(raw if raw.strip() else '(log vide)')
print('-' * 60)
print('Code retour :', rc)
print('Rappel : Character Accuracy X%  ->  CER = (100 - X)%')
